In [ ]:

# Import the required libararies
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr 
from glob import glob
import numpy as np
# Read-in the GEOTIFF image

files = sorted(glob(
    "/glade/campaign/acom/satellite/tropomi/no2/"
    "S5P_OFFL_L2__NO2____20240211T*.nc"
))

# sat_file = "/glade/campaign/acom/satellite/tropomi/no2/S5P_OFFL_L2__NO2____20240209T085706_20240209T103836_32773_03_020600_20240211T*.nc"

dataset = xr.open_dataset(sat_files[0], group = "PRODUCT")

# print(dataset)

no2 = dataset["nitrogendioxide_tropospheric_column"].squeeze()
qa = dataset["qa_value"].squeeze()
no2 = no2.where(np.isfinite(no2))
#no2 = no2.where(no2 > 0)
no2 = no2.where(qa > 0.75)

# Set figure parameters
fig = plt.figure(figsize=(10, 8), dpi=350)

plt.rcParams['font.size'] = 14
plt.rcParams['axes.linewidth'] = 1

# Plot the map
ax = plt.axes(projection=ccrs.PlateCarree())

# Define the image extent. These coordinates must be gotten from the google earth engine code
ax.set_extent([20, 110, -90, 90]) # [left_lon, right_lon, south_lat, north_lat

ax.set_facecolor("#dceef7")   # ocean

land = cfeature.NaturalEarthFeature(
    "physical",
    "land",
    "10m",
    facecolor="#efe7d3",
    edgecolor="none",
)

lakes = cfeature.NaturalEarthFeature(
    "physical",
    "lakes",
    "10m",
    facecolor="#dceef7",
    edgecolor="none",
)

ax.add_feature(land, zorder=0)
ax.add_feature(lakes, zorder=1)

vmin = np.nanpercentile(no2, 5)
vmax = np.nanpercentile(no2, 99)

pc = ax.pcolormesh(
    dataset.longitude.squeeze(),
    dataset.latitude.squeeze(), 
    no2, 
    cmap = "inferno",
    shading = "auto",
    vmin = vmin, 
    vmax = vmax,
)


# # Add geographical features to the map
# lakes_10m = cfeature.NaturalEarthFeature('physical','lakes','10m')
# states = cfeature.NaturalEarthFeature('cultural', scale="50m",
#                              facecolor="none",
#                              name="admin_1_states_provinces_lines")
# ax.add_feature(lakes_10m, facecolor='none', edgecolor='k')
# ax.add_feature(states, linewidth=0.5, edgecolor="black")

ax.coastlines(resolution='10m', color='black', linewidth=0.5)
ax.add_feature(cartopy.feature.BORDERS, linewidth=0.4)

# colorbar_axes_tro = plt.gcf().add_axes([0.2, 0.125, 0.6, 0.04])

# cb = plt.colorbar(plot_no2_tr, colorbar_axes_tro, 
#                   orientation='horizontal', 
#                   label='$\mathregular{NO_2}$ (molecules/$\mathregular{cm^2}$)')

plt.colorbar(pc, ax = ax, label='$\mathregular{NO_2}$ (molecules/$\mathregular{cm^2}$)')

plt.savefig(f"./tropomi_example.png", dpi=350, bbox_inches="tight")
plt.show()
